# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrabansal10/FlyRank_Internship/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
%run "./w05_model.ipynb"

from sklearn.base import clone

model = clone(fitted_models["Logistic regression"])
model.fit(X, y)

queue_scores = model.predict_proba(X)[:, 1]

Rows: 30000
Clients: 32
Proxy-label rate: 0.542
Safe feature-frame shape: (30000, 32)
Train rows: 23837
Held-out rows: 6163
Train clients: 25
Held-out clients: 7
Held-out base rate: 0.511
Held-out base rate: 0.511
Frozen baseline Precision@50: 0.42


,method,held_out_base_rate,precision_at_50
0,Logistic regression,0.511,0.84
1,Random forest,0.511,0.72
2,Decision tree (max_depth=3),0.511,0.56
3,Frozen rule baseline,0.511,0.42


Best Model : Logistic regression 
 (precision@50) :  0.84


,feature,importance
23,numeric__log_impressions_90d,1.005701
9,numeric__users_90d,0.917306
8,numeric__sessions_90d,0.741068
24,numeric__log_clicks_90d,0.589950
37,categorical__content_type_comparison article,0.484534
49,categorical__freshness_tier_181+,0.467656
32,numeric__missingindicator_avg_position,0.437308
22,numeric__has_position_data,0.437308
42,categorical__main_intent_navigational,0.359051
18,numeric__avg_position,0.296520


,content_type,main_intent,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update,actual_proxy_label,predicted_decline_score
27993,keyword article,informational,1266,0.0,4.6,106,106,0,0.932771
12869,keyword article,informational,15101,0.0,5.7,421,7,0,0.930318
26614,keyword article,informational,290,0.0,5.9,96,20,0,0.918188
25560,keyword article,informational,1463,0.0,1.5,106,8,0,0.916740
20736,keyword article,informational,3115,0.0,12.8,275,104,0,0.909005


In [10]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/rudrabansal10/FlyRank_Internship/43b468d73eba109085f02d01f3a59754d5356453/data/raw/content_refresh_anonymized.csv")

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [13]:
import numpy as np

queue = df[[
    "content_type",
    "main_intent",
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
]].copy()

queue["review_score"] = model.predict_proba(X)[:, 1]

queue["low_ctr_visible"] = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)

queue["reason_code"] = np.select(
    [
        queue["low_ctr_visible"],
        queue["impressions_90d"] >= 500,
    ],
    [
        "LOW_CTR_VISIBLE_PAGE",
        "HIGH_SEARCH_VISIBILITY",
    ],
    default="MODEL_REVIEW_CANDIDATE",
)

queue["recommended_action"] = np.select(
    [
        queue["low_ctr_visible"],
        queue["impressions_90d"] >= 500,
    ],
    [
        "Review title, metadata, and search snippet",
        "Review content relevance and search visibility",
    ],
    default="Review manually before any change",
)

queue = queue.sort_values("review_score", ascending=False).reset_index(drop=True)
queue["review_rank"] = queue.index + 1

display(queue.head(20))

,content_type,main_intent,impressions_90d,ctr,avg_position,days_since_last_update,review_score,low_ctr_visible,reason_code,recommended_action,review_rank
0,keyword article,commercial,4238,0.21,7.9,22,1.000000,True,LOW_CTR_VISIBLE_PAGE,"Review title, metadata, and search snippet",1
1,keyword article,informational,288426,0.92,4.8,20,0.999999,False,HIGH_SEARCH_VISIBILITY,Review content relevance and search visibility,2
2,keyword article,transactional,62927,3.40,7.2,20,0.999949,False,HIGH_SEARCH_VISIBILITY,Review content relevance and search visibility,3
3,keyword article,commercial,28192,4.78,9.1,20,0.999732,False,HIGH_SEARCH_VISIBILITY,Review content relevance and search visibility,4
4,keyword article,informational,112488,1.12,5.7,20,0.998571,False,HIGH_SEARCH_VISIBILITY,Review content relevance and search visibility,5
5,keyword article,informational,62841,1.18,19.8,20,0.995854,False,HIGH_SEARCH_VISIBILITY,Review content relevance and search visibility,6
6,keyword article,informational,167858,0.76,5.1,20,0.995825,False,HIGH_SEARCH_VISIBILITY,Review content relevance and search visibility,7
7,keyword article,informational,148515,0.81,5.0,20,0.994419,False,HIGH_SEARCH_VISIBILITY,Review content relevance and search visibility,8
8,keyword article,informational,79994,0.95,4.8,20,0.993131,False,HIGH_SEARCH_VISIBILITY,Review content relevance and search visibility,9
9,keyword article,informational,51341,1.21,5.9,20,0.992120,False,HIGH_SEARCH_VISIBILITY,Review content relevance and search visibility,10


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended use:
This queue helps content editors decide which pages to review first when capacity is limited.

Valid use:
Use the score and reason code to prioritize manual review of titles, metadata, relevance, freshness, and content quality.

Limits:
The model was evaluated on a starter proxy label, not a future outcome. It does not predict Google’s algorithm, guarantee future decline, or prove that refreshing a page will improve performance.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human review checks:
- Confirm the page is still strategically important.
- Check whether the page matches current search intent.
- Review title, metadata, and snippet before changing content.
- Check for outdated facts, broken links, and missing sections.
- Check whether a recent refresh or campaign explains the metrics.
- Compare against relevant competing pages before editing.

Never automate:
- Publishing or rewriting content automatically
- Deleting or redirecting pages automatically
- Changing titles or metadata automatically
- Treating the score as proof of future decline
- Claiming a refresh will cause recovery

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>frequency</th>
      <th>monitoring_check</th>
      <th>retrain_or_review_trigger</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>Monthly</td>
      <td>Review score distribution and number of high-p...</td>
      <td>Large unexpected shift in scores or queue size</td>
    </tr>
    <tr>
      <th>1</th>
      <td>Monthly</td>
      <td>Check CTR, impressions, and position patterns</td>
      <td>Signal patterns no longer resemble the trainin...</td>
    </tr>
    <tr>
      <th>2</th>
      <td>Quarterly</td>
      <td>Review editor outcomes for reviewed pages</td>
      <td>Reason codes no longer help reviewers prioriti...</td>
    </tr>
    <tr>
      <th>3</th>
      <td>When new time-series data is available</td>
      <td>Build earlier-feature/later-outcome validation</td>
      <td>Replace starter proxy model with future-lookin...</td>
    </tr>
  </tbody>
</table>
</div>

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [17]:
from pathlib import Path

output_dir = Path("../../work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

export_columns = [
    "review_rank",
    "review_score",
    "reason_code",
    "recommended_action",
    "content_type",
    "main_intent",
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
]

queue[export_columns].head(100).to_csv(
    output_dir / "ml10_review_queue_top100.csv",
    index=False,
)

print("Saved:", output_dir / "ml10_review_queue_top100.csv")

Saved: ../../work/outputs/ml10_review_queue_top100.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.